# Python 기초 통합 프로젝트
## Part 2. 판매 점검 코드를 함수화하고 결과 저장하기

### Part 1과의 연결

Part 1에서 작성한 판매금액 구간 분류와 배송 확인 거래 검색 코드를 함수로 바꾸어 반복 사용이 가능한 구조로 개선합니다.

### 실무 시나리오

영업관리팀은 매일 새로운 판매 데이터에 같은 점검 기준을 적용합니다. 데이터 담당자는 반복 코드를 함수로 정리하고, 배송 확인 대상 거래를 CSV 파일로 저장하여 담당자에게 전달해야 합니다.

### 과제 목표

- 반복 코드를 재사용 가능한 함수로 작성할 수 있습니다.
- 매개변수, 반환값, 기본값을 사용할 수 있습니다.
- `TypeError`, `ValueError`, `PermissionError` 등 오류 유형을 구분해 처리할 수 있습니다.
- 분석 결과를 CSV 파일로 저장하고 다시 확인할 수 있습니다.
- 심화 문제에서 고액 반품 결과를 JSON으로 저장할 수 있습니다.

### 사용 환경

- 결과물: `.ipynb` 파일 1개
- 필수 생성 파일: `delivery_check_sales.csv`
- 심화 생성 파일: `high_value_return_sales.json`
- 사용 언어: Python 3.X
- 사용 라이브러리: pandas

## 제공 코드. 데이터 불러오기

In [1]:
import pandas as pd

DATA_FILE = "자동차_판매_데이터.csv"

df = pd.read_csv(DATA_FILE)

sale_ids = df["SaleID"].tolist()
sale_amounts = df["FinalSaleAmount"].tolist()
delivery_days = df["DeliveryDays"].tolist()
stock_statuses = df["StockStatus"].tolist()
returned_flags = df["IsReturned"].tolist()

print("전체 거래 수:", len(df))
df.describe()

전체 거래 수: 500


,ModelYear,Quantity,UnitPrice,DiscountRate,FinalSaleAmount,DeliveryDays,CustomerRating
count,500.00000,500.000000,5.000000e+02,500.00000,5.000000e+02,500.000000,500.000000
mean,2024.48800,1.412000,3.609660e+07,0.05614,4.859311e+07,12.792000,4.138800
std,0.66237,0.920843,1.355850e+07,0.03632,3.900394e+07,7.420705,0.485764
min,2023.00000,1.000000,1.380000e+07,0.00000,1.292700e+07,1.000000,2.600000
25%,2024.00000,1.000000,2.690000e+07,0.03000,2.678400e+07,7.000000,3.800000
50%,2025.00000,1.000000,3.200000e+07,0.05000,3.295400e+07,12.000000,4.100000
75%,2025.00000,1.000000,4.430000e+07,0.07000,5.640750e+07,17.000000,4.500000
max,2025.00000,5.000000,6.830000e+07,0.15000,2.886400e+08,35.000000,5.000000


# 문제 1. 판매금액 구간 분류 함수 만들기

## 함수

```python
classify_sales_by_amount(sale_ids, sale_amounts)
```

## 요구사항

1. 두 입력값이 리스트가 아니면 `TypeError`를 발생시킵니다.
2. 두 리스트의 길이가 다르면 `ValueError`를 발생시킵니다.
3. 데이터가 비어 있으면 `ValueError`를 발생시킵니다.
4. 고가·중가·일반 거래의 `SaleID`를 딕셔너리로 반환합니다.
5. 함수를 호출하고 결과를 출력합니다.

In [13]:
# 문제 1 코드를 작성하세요.
import pandas as pd

DATA_FILE = "자동차_판매_데이터.csv"

df = pd.read_csv(DATA_FILE)

sale_ids = df["SaleID"].tolist()
sale_amounts = df["FinalSaleAmount"].tolist()
delivery_days = df["DeliveryDays"].tolist()
stock_statuses = df["StockStatus"].tolist()
returned_flags = df["IsReturned"].tolist()

print(len(sale_ids))
print(len(sale_amounts))

high_sale_ids = []
middle_sale_ids = []
nrml_sale_ids = []


def classify_sales_by_amount(sale_ids, sale_amounts):
    if type(sale_ids) != list or type(sale_amounts) != list:
        raise TypeError('입력값은 리스트 형식이어야 합니다.')
    if len(sale_ids) == 0 or len(sale_amounts) == 0:
        raise ValueError('두 리스트의 길이가 다릅니다.')
    if len(sale_ids) != len(sale_amounts):
        raise ValueError('데이터가 없습니다.')

    classified_sales = {
        '고가 거래': [],
        '중가 거래': [],
        '일반 거래': []}

    for i in range(len(sale_ids)):
            sale_amount = sale_amounts[i]
            sale_id = sale_ids[i]
    
            if sale_amount >= 70000000:
                classified_sales['고가 거래'].append((sale_id, sale_amount))
    
            elif sale_amount >= 40000000:
                classified_sales['중가 거래'].append((sale_id, sale_amount))
    
            else:
                classified_sales['일반 거래'].append((sale_id, sale_amount))
    return classified_sales

result = classify_sales_by_amount(sale_ids, sale_amounts)
print(result)
print(len(result['고가 거래']))
print(len(result['중가 거래']))
print(len(result['일반 거래']))

500
500
{'고가 거래': [('S2025010064', 85746000), ('S2025010264', 192000000), ('S2025010093', 90356000), ('S2025010135', 113460000), ('S2025010483', 118800000), ('S2025010165', 288640000), ('S2025020115', 101724000), ('S2025020161', 163494000), ('S2025020030', 111600000), ('S2025020413', 115400000), ('S2025020437', 127300000), ('S2025020031', 90356000), ('S2025030414', 77900000), ('S2025030139', 199240000), ('S2025030005', 149340000), ('S2025030049', 111135000), ('S2025040389', 70000000), ('S2025040494', 178695000), ('S2025040336', 165870000), ('S2025040344', 91140000), ('S2025040284', 93000000), ('S2025040369', 84672000), ('S2025040429', 232940000), ('S2025040262', 105120000), ('S2025040288', 172854000), ('S2025040206', 142396000), ('S2025040178', 228760000), ('S2025050451', 82840000), ('S2025050444', 110580000), ('S2025050127', 230472000), ('S2025050303', 92100000), ('S2025050183', 110856000), ('S2025050260', 168795000), ('S2025050273', 152000000), ('S2025050401', 174636000), ('S20250600

# 문제 2. 배송 확인 거래 검색 함수 만들기

## 함수

```python
find_delivery_check_sales(
    sale_ids,
    delivery_days,
    stock_statuses
)
```

## 요구사항

1. 세 입력값이 리스트가 아니면 `TypeError`를 발생시킵니다.
2. 세 리스트의 길이가 다르면 `ValueError`를 발생시킵니다.
3. 배송일 20일 이상이면서 출고완료가 아닌 거래의 `SaleID`를 반환합니다.
4. 정상 입력과 잘못된 입력을 각각 테스트합니다.
5. `TypeError`와 `ValueError`를 별도의 `except`에서 처리합니다.

In [20]:
import pandas as pd

df = pd.read_csv("자동차_판매_데이터.csv")

for col in df.columns:
    print(f"=== [컬럼명: {col}] ===")
    print(f"고유값 개수: {df[col].nunique()}개")
    print(f"값 목록: {df[col].unique()[:10]}")  
    print()

=== [컬럼명: SaleID] ===
고유값 개수: 500개
값 목록: <ArrowStringArray>
['S2025010447', 'S2025010064', 'S2025010207', 'S2025010264', 'S2025010326',
 'S2025010497', 'S2025010129', 'S2025010416', 'S2025010335', 'S2025010098']
Length: 10, dtype: str

=== [컬럼명: SaleDate] ===
고유값 개수: 267개
값 목록: <ArrowStringArray>
['2025-01-01', '2025-01-02', '2025-01-04', '2025-01-05', '2025-01-06',
 '2025-01-09', '2025-01-10', '2025-01-13', '2025-01-14', '2025-01-15']
Length: 10, dtype: str

=== [컬럼명: Manufacturer] ===
고유값 개수: 6개
값 목록: <ArrowStringArray>
['르노코리아', '현대', '쉐보레', '제네시스', '기아', 'KG모빌리티']
Length: 6, dtype: str

=== [컬럼명: Model] ===
고유값 개수: 19개
값 목록: <ArrowStringArray>
['XM3', '아이오닉5', '말리부', 'G70', 'EV6', '스포티지', '트랙스 크로스오버', 'K3', '투싼', 'G80']
Length: 10, dtype: str

=== [컬럼명: VehicleType] ===
고유값 개수: 2개
값 목록: <ArrowStringArray>
['SUV', 'Sedan']
Length: 2, dtype: str

=== [컬럼명: FuelType] ===
고유값 개수: 5개
값 목록: <ArrowStringArray>
['Gasoline', 'Electric', 'Hybrid', 'Diesel', 'LPG']
Length: 5, dtype: str

=== 

In [34]:
# 문제 2 코드를 작성하세요.
import pandas as pd

DATA_FILE = "자동차_판매_데이터.csv"

df = pd.read_csv(DATA_FILE)

sale_ids = df["SaleID"].tolist()
sale_amounts = df["FinalSaleAmount"].tolist()
delivery_days = df["DeliveryDays"].tolist()
stock_statuses = df["StockStatus"].tolist()
returned_flags = df["IsReturned"].tolist()

def find_delivery_check_sales(
    sale_ids,
    delivery_days,
    stock_statuses):
    
    if not (type(sale_ids) is list and type(delivery_days) is list and type(stock_statuses) is list):
        raise TypeError('입력값은 리스트 형식이어야 합니다.')
    if not (len(sale_ids) == len(delivery_days) == len(stock_statuses)):
        raise ValueError('세 리스트의 길이가 다릅니다.')

    d_sale_ids = []
    
    for i in range(len(sale_ids)):
        d_sale_id = sale_ids[i]
        d_delivery_day = delivery_days[i]
        d_stock_status = stock_statuses[i]

        if d_delivery_day >= 20 and d_stock_status != '출고완료':
                d_sale_ids.append(d_sale_id)

    return d_sale_ids    


print("=== [테스트 1] 정상 입력 ===")
try:
    sample_sale_ids = [
        "S2025010447", "S2025010064", "S2025010207", 
        "S2025010264", "S2025010326", "S2025010497"]
    sample_delivery_days = [21, 22, 11, 5, 32, 35]
    sample_stock_statuses = ["출고완료", "재고보유", "출고완료", "주문생산", "주문생산", "출고완료"]

    result = find_delivery_check_sales(sample_sale_ids, sample_delivery_days, sample_stock_statuses)
    print("검색 결과:", result)
    
except TypeError:
    print('입력값은 리스트 형식이어야 합니다.')
except ValueError:
    print('세 리스트의 길이가 다릅니다.')
finally:
    print("검색이 완료되었습니다.")



print("\n=== [테스트 2] 비정상 입력 ===")
try:
    sampe_sale_ids = ["S2025010447", "S2025010064", "S2025010326"]
    sample_delivery_days = [21, 22, 32, 35]
    sample_stock_statuses = ["출고완료", "재고보유", "주문생산"]

    result_wrong = find_delivery_check_sales(sample_sale_ids, sample_delivery_days, sample_stock_statuses)
    print("검색 결과:", result_wrong)

except TypeError:
    print('입력값은 리스트 형식이어야 합니다.')
except ValueError:
    print('세 리스트의 길이가 다릅니다.')
finally:
    print("검색이 완료되었습니다.")

=== [테스트 1] 정상 입력 ===
검색 결과: ['S2025010064', 'S2025010326']
검색이 완료되었습니다.

=== [테스트 2] 비정상 입력 ===
세 리스트의 길이가 다릅니다.
검색이 완료되었습니다.


# 문제 3. 배송 점검 결과를 CSV로 저장하기

## 요구사항

1. 문제 2에서 반환된 `SaleID`로 원본 DataFrame의 거래를 선택합니다.
2. `delivery_check_sales.csv`로 저장합니다.
3. 파일 인덱스는 저장하지 않습니다.
4. 저장 파일을 다시 불러와 거래 수를 비교합니다.
5. `PermissionError`와 그 외 `OSError`를 구분하여 처리합니다.

In [36]:
# 문제 3 코드를 작성하세요.
d_sale_ids = find_delivery_check_sales(df['SaleID'].tolist(), df['DeliveryDays'].tolist(), df['StockStatus'].tolist())

csv_filename = "delivery_check_sales.csv"

try:
    delivery_check_df = df[df['SaleID'].isin(d_sale_ids)]
    print(f'추출된 배송 거래 수: {len(delivery_check_df)}건')

    delivery_check_df.to_csv(csv_filename, index=False, encoding='utf-8')
    print(f'"{csv_filename}" 파일 저장되었습니다.')

    reloaded_df = pd.read_csv(csv_filename)
    reloaded_count = len(reloaded_df)
    original_count = len(delivery_check_df)

    print(f'추출된 파일의 거래 수: {reloaded_count}건')
    if original_count == reloaded_count:
        print("거래 수 일치")
    else:
        print("거래 수 불일치")

except PermissionError:
    print('권한이 없습니다.')
except OSError:
    print('OS에러가 발생했습니다.')


추출된 배송 거래 수: 48건
"delivery_check_sales.csv" 파일 저장되었습니다.
추출된 파일의 거래 수: 48건
거래 수 일치


# 심화 문제. 고액 반품 검색 및 JSON 저장

이 문제는 **5점 수준을 위한 선택 문제**입니다.

## 요구사항

1. `find_high_value_returns()` 함수를 작성합니다.
2. `min_amount=70_000_000` 기본값을 사용합니다.
3. 입력 자료형이 잘못되면 `TypeError`, 기준값이 잘못되면 `ValueError`를 발생시킵니다.
4. 고액 반품 거래를 JSON 파일로 저장합니다.
5. 저장한 JSON을 다시 읽어 거래 수를 확인합니다.
6. `PermissionError`, `OSError`, `ValueError`를 구분해 처리합니다.

In [5]:
# 심화 문제 코드를 작성하세요.
import json
import os
import pandas as pd

DATA_FILE = "자동차_판매_데이터.csv"

df = pd.read_csv(DATA_FILE)

sale_ids = df["SaleID"].tolist()
sale_amounts = df["FinalSaleAmount"].tolist()
delivery_days = df["DeliveryDays"].tolist()
stock_statuses = df["StockStatus"].tolist()
returned_flags = df["IsReturned"].tolist()

def find_high_value_returns(sale_ids, final_sale_amounts, is_returned_list, min_amount=70_000_000):

    if not (type(sale_ids) is list
        and type(final_sale_amounts) is list
        and type(is_returned_list) is list):
        raise TypeError('입력값은 리스트 형태여야 합니다.')

    if type(min_amount) not in (int, float):
        raise ValueError('기준 금액은 숫자여야 합니다.')

    if not ((len(sale_ids)) == len(final_sale_amounts) == len(is_returned_list)):
        raise ValueError('세 리스트의 길이가 다릅니다.')

    high_value_returns = []
 
    for i in range(len(sale_ids)):
        rhv_sale_id = sale_ids[i]
        rhv_sale_amount = sale_amounts[i]
        rhv_returned_flag = returned_flags[i]

        if rhv_sale_amount >= min_amount and rhv_returned_flag == 'Y':
            high_value_returns.append({
                'SaleID': rhv_sale_id,
                'FinalSaleAmount': rhv_sale_amount,
                'IsReturned': rhv_returned_flag})
            
    return high_value_returns


json_filename = 'high_value_returns.json'

try:
    sale_ids = df["SaleID"].tolist()
    final_amounts = df["FinalSaleAmount"].tolist()
    is_returned_list = df["IsReturned"].tolist()

    return_records = find_high_value_returns(
        sale_ids, 
        final_amounts, 
        is_returned_list
    )
    print(f"추출된 고액 반품 거래 수: {len(return_records)}건")

    with open(json_filename, 'w', encoding='utf-8') as f:
        print(f'"{json_filename}" 파일 저장 완료')

    with open(json_filename, "r", encoding="utf-8") as f:
        loaded_records = json.load(f)

    records_count = len(loaded_records)
    print(len(records_count))

    if len(return_records) == records_count:
        print('확인 성공: 추출 데이터와 JSON 데이터 수가 일치합니다.')

    else:
        print('확인 실패: 데이터 수가 일치하지 않습니다.')

except ValueError:
    print('입력값 또는 기준값이 올바르지 않습니다.')
except PermissionError:
    print('권한이 없습니다.')
except OSError:
    print('OS 오류가 발생했습니다.')
except TypeError:
    print('자료형이 올바르지 않습니다.')

추출된 고액 반품 거래 수: 3건
"high_value_returns.json" 파일 저장 완료
입력값 또는 기준값이 올바르지 않습니다.


# 제출 결과물

| 결과물 | 구분 |
|---|---|
| 판매금액 구간 분류 함수 | 필수 |
| 배송 확인 거래 함수 | 필수 |
| `TypeError`, `ValueError` 구분 처리 | 필수 |
| 배송 확인 CSV 저장 및 재확인 | 필수 |
| `PermissionError`, `OSError` 구분 처리 | 필수 |
| 고액 반품 함수와 JSON 저장 | 심화 |